<a href="https://colab.research.google.com/github/nupurmadaan04/Machine-learning-notebooks/blob/main/FUNCTION%20TRANSFORMATION%20%2B%20FEATURE%20SELECTION%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("titanic.csv")

FUNCTION TRANSFORMATION

In [3]:
print(df['Fare'].skew())

3.6872133081121405


In [9]:
# function transformer - non normal distribution to normal distribution

# by default its function is np.log1p
# np.log1p(3) = log(1+3)= log(4) = value of log4

from sklearn.preprocessing import FunctionTransformer
import numpy as np

# create transformer (log1p for right skew)
ft = FunctionTransformer(np.log1p)

# apply on Fare column (must be 2D → double brackets)
df['Fare_log'] = ft.fit_transform(df[['Fare']])

In [8]:
print(df['Fare_log'].skew())

0.8621677993028847


In [ ]:
# we can use custom function using lambda and then enter custom function according to data
from sklearn.preprocessing import FunctionTransformer
import numpy as np

ft = FunctionTransformer(lambda x: np.log1p(x))

df['Fare_log'] = ft.fit_transform(df[['Fare']])

In [10]:
# if the data is right skewed, right tailed, more huge values spread then we have to compress data mostly used is np.log1p
# if the data is left skewed, left tailed, more small values spread then we can increase value by using sqaure,cube or reflex log

FEATURE SELECTION



In [11]:
# X input
# y is targeted output
X = df[['Age', 'Fare', 'Pclass']]
y = df['Survived']

In [15]:
# FILTER SELECTION, SELECT CLOSELY RELATED COLUMNS AND DROP FAR AWAY COLUMNS
corrr = df.select_dtypes(include=np.number).corr()
# numeric columns select karke unka correlation matrix banaya

target_corr = corrr['Survived'].sort_values(ascending=False)
# target (Survived) ke saath sab features ka correlation nikala aur sort kiya

filter_features = target_corr[abs(target_corr) > 0.1].index
# threshold lagaya (|corr| > 0.1) → strong features select kiye

In [23]:
# WRAPPER METHOD → tries combinations of features using model

# FORWARD ELIMINATION
from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer # Import SimpleImputer

# Model define kiya (classification problem ke liye)
model = LogisticRegression(random_state=42, solver='liblinear')

# Impute missing values in X
imputer = SimpleImputer(strategy='mean')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns) # Impute and convert back to DataFrame

# FORWARD SELECTION
sfs_forward = SequentialFeatureSelector(
    model,
    k_features=2,      # final me 2 best features select karega
    forward=True,      # True → forward selection (add features)
    cv=2,              # cross-validation (model ko robust banata hai)
    n_jobs=-1          # all CPU cores use karega (fast)
)

# model ko train karke best features choose karega
sfs_forward.fit(X_imputed, y) # Fit with the imputed data

# selected feature names
print("Forward Selected Features:", sfs_forward.k_feature_names_)

Forward Selected Features: ('Age', 'Fare')


In [24]:
# BACKWARD SELECTION
sfs_backward = SequentialFeatureSelector(
    model,
    k_features=2,      # final me 2 best features select karega
    forward=False,     # False → backward selection (remove features)
    cv=2,              # cross-validation (model ko robust banata hai)
    n_jobs=-1          # all CPU cores use karega (fast)
)

# model ko train karke best features choose karega
sfs_backward.fit(X_imputed, y)

# selected feature names
print("Backward Selected Features:", sfs_backward.k_feature_names_)

Backward Selected Features: ('Age', 'Fare')


In [ ]:
# embedded

In [26]:
# Embedded method = feature selection happens automatically during model training
from sklearn.linear_model import LogisticRegression

# Logistic Regression model with L1 penalty (Lasso)
# L1 penalty kuch coefficients ko 0 bna deta h → feature selection hota h
# C = inverse of regularization strength (chota C → zyada penalty → zyada features remove)
lasso_model = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, random_state=42)

# Model ko train kro (X_imputed = missing values handled data)
lasso_model.fit(X_imputed, y)

# Hr feature ka coefficient nikalo
# coef_[0] → kyunki binary classification h (single output)
coefficients = pd.Series(lasso_model.coef_[0], index=X_imputed.columns)

print("Feature Coefficients (Lasso Logistic Regression):")
print(coefficients)

# Jo features ka coefficient 0 nahi h → vhi important features h
selected_features_lasso = coefficients[coefficients != 0].index.tolist()

print("\nFeatures selected by Lasso:", selected_features_lasso)

Feature Coefficients (Lasso Logistic Regression):
Age      -0.020244
Fare      0.007642
Pclass   -0.084133
dtype: float64

Features selected by Lasso: ['Age', 'Fare', 'Pclass']
